# Assignment 1 model — retrain for Assignment 2B (RAG)

The RAG assignment (2B) uses the **Assignment 1 fine-tuned model as the answer generator**. The trained adapters from Assignment 1 were not kept, so this notebook retrains the same model, with the same data and hyper-parameters, and **saves it to Google Drive** so the RAG notebook can load it.

```text
Qwen/Qwen3-0.6B-Base ──CPT (QLoRA r=64, 3 epochs, raw Constitution text)──► cpt-merged (fp16)
cpt-merged           ──SFT (QLoRA r=16, 3 epochs, 200 Q&A pairs)────────► sft-on-cpt-adapter
```

| Setting | CPT | SFT |
|---|---|---|
| Data | 500 train sections, packed into 1024-token blocks (18 held-out articles for eval) | 200 train pairs (+22 validation) from `instructions.jsonl` |
| LoRA | r=64, α=128, dropout 0.05, all attention + MLP projections | r=16, α=32, dropout 0.05, same modules |
| Optimisation | 3 epochs, batch 2 × grad-accum 8, lr 2e-4, cosine, 10 warmup steps | 3 epochs, batch 8, lr 2e-4, weight decay 0.01 |
| Base quantisation | 4-bit NF4, double quant (QLoRA) | same |

This is a straight reproduction of `constitution_cpt_qlora_final.ipynb` (Assignment 1). It skips the PDF extraction, because `sections.jsonl` already holds the cleaned and segmented text (it rebuilds `corpus_clean.txt` exactly), and it skips the base→SFT ablation, which 2B does not need.

**Expected runtime on a free T4:** about 15 minutes (CPT ~9 min, merge ~1 min, SFT ~3 min, sanity eval ~2 min).

### Before you run
1. **Runtime → Change runtime type → T4 GPU**.
2. In Google Drive, create the folder `MyDrive/llm_assignment_2/data/` and upload these files from `llm_assingment_1/`:
   `sections.jsonl`, `instructions.jsonl`, `corpus_clean.txt`.
3. Run all cells. Each stage is skipped if its output already exists on Drive, so if Colab disconnects, just run again.

**Output on Drive (`MyDrive/llm_assignment_2/models/`):** `cpt-adapter/`, `cpt-merged/`, `sft-on-cpt-adapter/`, `a1_model_info.json`.

In [1]:
# ── Paths: Google Drive on Colab, a local folder for the CPU smoke test ──
import os, sys, json, time, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
SMOKE_TEST = os.environ.get("SMOKE_TEST") == "1"     # tiny CPU run used only to test this notebook end-to-end

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/llm_assignment_2")
else:
    PROJECT_DIR = Path(os.environ.get("PROJECT_DIR", ".")).resolve()

DATA_DIR = PROJECT_DIR / "data"
MODELS_DIR = PROJECT_DIR / ("models_smoke" if SMOKE_TEST else "models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)
FORCE_RETRAIN = False        # True = ignore what is already on Drive and train again

missing = [f for f in ["sections.jsonl", "instructions.jsonl"] if not (DATA_DIR / f).exists()]
assert not missing, f"Upload {missing} (from llm_assingment_1/) to {DATA_DIR} and re-run this cell."
print("project dir:", PROJECT_DIR, "| models dir:", MODELS_DIR, "| smoke test:", SMOKE_TEST)

Mounted at /content/drive
project dir: /content/drive/MyDrive/llm_assignment_2 | models dir: /content/drive/MyDrive/llm_assignment_2/models | smoke test: False


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# ── Libraries (Colab already has torch / transformers / accelerate) ──
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "bitsandbytes>=0.46.1", "peft"], check=True)
    # Assignment 1 hit an incompatible pre-installed torchao when merging adapters; upgrade it only if it is old.
    from importlib.metadata import version as _v, PackageNotFoundError
    from packaging.version import Version
    try:
        if Version(_v("torchao")) < Version("0.16.0"):
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "torchao>=0.16.0"], check=True)
    except PackageNotFoundError:
        pass

In [4]:
import gc, math, random, re, collections, shutil
import torch
import transformers
from packaging import version
from transformers import (AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, Trainer,
                          TrainingArguments, default_data_collator, set_seed)
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training

set_seed(42)
MODEL_ID = "Qwen/Qwen3-0.6B-Base"
HAS_GPU = torch.cuda.is_available()
QUANTIZE = HAS_GPU                                                     # bitsandbytes 4-bit needs CUDA
USE_BF16 = HAS_GPU and torch.cuda.get_device_capability()[0] >= 8     # T4 (7.5) → fp16
COMPUTE_DTYPE = torch.bfloat16 if USE_BF16 else (torch.float16 if HAS_GPU else torch.float32)
DTYPE_KW = "dtype" if version.parse(transformers.__version__) >= version.parse("4.56") else "torch_dtype"

CPT_ADAPTER_DIR = MODELS_DIR / "cpt-adapter"
CPT_MERGED_DIR = MODELS_DIR / "cpt-merged"
SFT_ADAPTER_DIR = MODELS_DIR / "sft-on-cpt-adapter"
LOCAL_RUN_DIR = Path("/content/runs" if IN_COLAB else "runs")         # trainer scratch space, kept off Drive

print(f"transformers {transformers.__version__} | torch {torch.__version__} | GPU: "
      f"{torch.cuda.get_device_name(0) if HAS_GPU else 'none'} | 4-bit: {QUANTIZE} | compute dtype: {COMPUTE_DTYPE}")
if not HAS_GPU and not SMOKE_TEST:
    raise SystemExit("No GPU found: switch the runtime to a T4 GPU (Runtime → Change runtime type).")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
EOS = tokenizer.eos_token

def load_model(path, for_training=False):
    """Load a causal LM in 4-bit NF4 on GPU (QLoRA), full precision on CPU (smoke test). Same as Assignment 1."""
    kwargs = {DTYPE_KW: COMPUTE_DTYPE}
    if QUANTIZE:
        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=COMPUTE_DTYPE)
        kwargs["device_map"] = {"": 0}
    model = AutoModelForCausalLM.from_pretrained(path, **kwargs)
    model.config.use_cache = not for_training
    if for_training and QUANTIZE:
        model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True,
                                                gradient_checkpointing_kwargs={"use_reentrant": False})
    return model

def free_memory():
    """Call after `del model, ...` so the GPU memory is really released."""
    gc.collect()
    if HAS_GPU:
        torch.cuda.empty_cache()

LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

def training_args(out_dir, **overrides):
    args = dict(output_dir=str(out_dir), logging_steps=2, save_strategy="no", report_to="none",
                eval_strategy="epoch", lr_scheduler_type="cosine", warmup_steps=10,
                optim="paged_adamw_8bit" if QUANTIZE else "adamw_torch",
                fp16=HAS_GPU and not USE_BF16, bf16=USE_BF16,
                gradient_checkpointing=QUANTIZE,
                gradient_checkpointing_kwargs={"use_reentrant": False} if QUANTIZE else None, seed=42)
    args.update(overrides)
    return TrainingArguments(**args)

def done(path):
    """A stage is complete when its saved config exists on Drive."""
    p = Path(path)
    return not FORCE_RETRAIN and ((p / "config.json").exists() or (p / "adapter_config.json").exists())

W1108 05:09:47.621000 1267 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1108 05:09:47.752000 1267 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


transformers 5.22.1 | torch 2.14.1+cu130 | GPU: Tesla T4 | 4-bit: True | compute dtype: torch.float16


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.68k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

## 1. Data — rebuild the Assignment 1 CPT documents and SFT splits

In [5]:
sections = [json.loads(l) for l in open(DATA_DIR / "sections.jsonl")]
pairs = [json.loads(l) for l in open(DATA_DIR / "instructions.jsonl")]

def as_document(s):
    """Every CPT document carries a short provenance header (identical to Assignment 1)."""
    return f"The Constitution of India — {s['part']}\n{s['id']}: {s['title']}\n\n{s['text']}" \
        if s["kind"] == "article" else f"The Constitution of India — {s['part']}\n\n{s['text']}"

if (DATA_DIR / "corpus_clean.txt").exists():
    assert "\n\n".join(as_document(s) for s in sections) == (DATA_DIR / "corpus_clean.txt").read_text(), \
        "sections.jsonl does not reproduce corpus_clean.txt"
    print("✓ sections.jsonl reproduces corpus_clean.txt exactly")

cpt_train_docs = [as_document(s) for s in sections if s["split"] == "train"]
cpt_heldout_docs = [as_document(s) for s in sections if s["split"] == "holdout"]
train_pairs = [p for p in pairs if p["split"] == "train"]
eval_pairs = [p for p in pairs if p["split"] == "eval"]
print(f"CPT docs: train {len(cpt_train_docs)} | held-out {len(cpt_heldout_docs)}")
print(f"Q&A pairs: train {len(train_pairs)} | eval {len(eval_pairs)}")

✓ sections.jsonl reproduces corpus_clean.txt exactly
CPT docs: train 500 | held-out 18
Q&A pairs: train 222 | eval 39


In [6]:
@torch.no_grad()
def perplexity(model, texts, block=1024, max_tokens=60_000):
    model.eval()
    ids = tokenizer(f"{EOS}".join(texts), add_special_tokens=False)["input_ids"][:max_tokens]
    nll, count = 0.0, 0
    for i in range(0, len(ids) - 1, block):
        chunk = torch.tensor([ids[i:i + block]], device=model.device)
        if chunk.shape[1] < 2:
            continue
        out = model(input_ids=chunk, labels=chunk)
        n = chunk.shape[1] - 1
        nll += out.loss.float().item() * n; count += n
    return math.exp(nll / count)

PPL_BLOCK = 128 if SMOKE_TEST else 1024
ppl_texts = cpt_heldout_docs[:2] if SMOKE_TEST else cpt_heldout_docs
metrics = {}

## 2. Continual pre-training (CPT) → merge into `cpt-merged`

In [7]:
BLOCK_SIZE = 128 if SMOKE_TEST else 1024

def pack(texts, block):
    ids = []
    for t in texts:
        ids += tokenizer(t, add_special_tokens=False)["input_ids"] + [tokenizer.eos_token_id]
    n = len(ids) // block
    blocks = [ids[i * block:(i + 1) * block] for i in range(n)]
    return [{"input_ids": b, "attention_mask": [1] * block, "labels": b} for b in blocks], len(ids)

if done(CPT_ADAPTER_DIR):
    print(f"CPT adapter already on Drive ({CPT_ADAPTER_DIR}), skipping CPT training.")
else:
    cpt_train_blocks, n_train_tok = pack(cpt_train_docs, BLOCK_SIZE)
    cpt_eval_blocks, n_eval_tok = pack(cpt_heldout_docs, BLOCK_SIZE)
    if SMOKE_TEST:
        cpt_train_blocks, cpt_eval_blocks = cpt_train_blocks[:4], cpt_eval_blocks[:1]
    print(f"CPT tokens: train {n_train_tok:,} → {len(cpt_train_blocks)} blocks of {BLOCK_SIZE} | "
          f"held-out {n_eval_tok:,} → {len(cpt_eval_blocks)} blocks")

    model = load_model(MODEL_ID, for_training=True)
    model = get_peft_model(model, LoraConfig(r=64, lora_alpha=128, lora_dropout=0.05, target_modules=LORA_TARGETS,
                                             bias="none", task_type="CAUSAL_LM"))
    model.print_trainable_parameters()
    cpt_trainer = Trainer(
        model=model,
        args=training_args(LOCAL_RUN_DIR / "cpt-run", num_train_epochs=1 if SMOKE_TEST else 3,
                           per_device_train_batch_size=2, per_device_eval_batch_size=2,
                           gradient_accumulation_steps=1 if SMOKE_TEST else 8, learning_rate=2e-4, weight_decay=0.0),
        train_dataset=cpt_train_blocks, eval_dataset=cpt_eval_blocks, data_collator=default_data_collator)
    t0 = time.time()
    cpt_trainer.train()
    metrics["cpt_train_minutes"] = (time.time() - t0) / 60
    print(f"CPT finished in {metrics['cpt_train_minutes']:.1f} min")
    model.save_pretrained(CPT_ADAPTER_DIR)
    del model, cpt_trainer; free_memory()

CPT tokens: train 157,436 → 153 blocks of 1024 | held-out 4,494 → 4 blocks


model.safetensors: reconstructing file:   0%|          |  0.00B / 1.19GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

trainable params: 40,370,176 || all params: 636,420,096 || trainable%: 6.3433


Epoch,Training Loss,Validation Loss
1,1.507324,1.349439
2,1.298179,1.211067
3,1.202778,1.178494


CPT finished in 5.5 min


In [8]:
if done(CPT_MERGED_DIR):
    print(f"Merged CPT model already on Drive ({CPT_MERGED_DIR}), skipping merge.")
else:
    merge_base = AutoModelForCausalLM.from_pretrained(MODEL_ID, **{DTYPE_KW: torch.float16 if HAS_GPU else torch.float32},
                                                      device_map={"": 0} if HAS_GPU else None)
    merged = PeftModel.from_pretrained(merge_base, CPT_ADAPTER_DIR).merge_and_unload()
    merged.save_pretrained(CPT_MERGED_DIR)
    tokenizer.save_pretrained(CPT_MERGED_DIR)
    del merge_base, merged; free_memory()
    print("saved", CPT_MERGED_DIR)

# Sanity check: CPT should lower perplexity on the 18 held-out articles (Assignment 1: 5.46 → 3.45).
for name, path in [("base", MODEL_ID), ("cpt", CPT_MERGED_DIR)]:
    m = load_model(path)
    metrics[f"ppl_heldout_{name}"] = perplexity(m, ppl_texts, PPL_BLOCK)
    del m; free_memory()
print(f"held-out Constitution perplexity: base {metrics['ppl_heldout_base']:.2f} → CPT {metrics['ppl_heldout_cpt']:.2f}")

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

saved /content/drive/MyDrive/llm_assignment_2/models/cpt-merged


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

held-out Constitution perplexity: base 5.46 → CPT 3.44


## 3. Instruction fine-tuning (SFT) on top of the CPT model

The prompt format is the same as in Assignment 1. The RAG notebook reuses it, adding a `### Context:` block with the retrieved chunks.

In [9]:
SYSTEM = ("You are an expert on the Constitution of India. Answer the question accurately and concisely "
          "in your own words.")

def build_prompt(instruction):
    return f"### System:\n{SYSTEM}\n\n### Question:\n{instruction}\n\n### Answer:\n"

MAX_LEN = 512
def encode_sft(pair):
    prompt_ids = tokenizer(build_prompt(pair["instruction"]), add_special_tokens=False)["input_ids"]
    answer_ids = tokenizer(pair["response"], add_special_tokens=False)["input_ids"] + [tokenizer.eos_token_id]
    ids = (prompt_ids + answer_ids)[:MAX_LEN]
    labels = ([-100] * len(prompt_ids) + answer_ids)[:MAX_LEN]
    return {"input_ids": ids, "attention_mask": [1] * len(ids), "labels": labels}

def sft_collator(batch):
    """Right-pad a batch; padded positions get label -100 so they never contribute to the loss."""
    width = max(len(b["input_ids"]) for b in batch)
    pad = lambda seq, v: seq + [v] * (width - len(seq))
    return {"input_ids": torch.tensor([pad(b["input_ids"], tokenizer.pad_token_id) for b in batch]),
            "attention_mask": torch.tensor([pad(b["attention_mask"], 0) for b in batch]),
            "labels": torch.tensor([pad(b["labels"], -100) for b in batch])}

shuffled = random.Random(0).sample(train_pairs, len(train_pairs))       # same split as Assignment 1
sft_val_pairs, sft_train_pairs = shuffled[:22], shuffled[22:]
sft_train, sft_val = [encode_sft(p) for p in sft_train_pairs], [encode_sft(p) for p in sft_val_pairs]
if SMOKE_TEST:
    sft_train, sft_val = sft_train[:8], sft_val[:2]
print(f"SFT train {len(sft_train)} | val {len(sft_val)} | longest example {max(len(e['input_ids']) for e in sft_train)} tokens")

if done(SFT_ADAPTER_DIR):
    print(f"SFT adapter already on Drive ({SFT_ADAPTER_DIR}), skipping SFT training.")
else:
    model = load_model(CPT_MERGED_DIR, for_training=True)
    model = get_peft_model(model, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, target_modules=LORA_TARGETS,
                                             bias="none", task_type="CAUSAL_LM"))
    model.print_trainable_parameters()
    trainer = Trainer(
        model=model,
        args=training_args(LOCAL_RUN_DIR / "sft-cpt-run", num_train_epochs=1 if SMOKE_TEST else 3,
                           per_device_train_batch_size=8, per_device_eval_batch_size=8,
                           gradient_accumulation_steps=1, learning_rate=2e-4, weight_decay=0.01),
        train_dataset=sft_train, eval_dataset=sft_val, data_collator=sft_collator)
    t0 = time.time()
    trainer.train()
    metrics["sft_train_minutes"] = (time.time() - t0) / 60
    print(f"SFT finished in {metrics['sft_train_minutes']:.1f} min")
    model.save_pretrained(SFT_ADAPTER_DIR)
    del model, trainer; free_memory()

SFT train 200 | val 22 | longest example 128 tokens


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

trainable params: 10,092,544 || all params: 606,142,464 || trainable%: 1.6650


Epoch,Training Loss,Validation Loss
1,2.207761,2.283095
2,1.764655,2.241585
3,1.579209,2.271083


SFT finished in 1.1 min


## 4. Sanity check — does the retrained model match Assignment 1?

The model is loaded exactly as the RAG notebook will load it: `cpt-merged` in 4-bit with the SFT adapter on top. It answers the 39 held-out evaluation questions, and we compare ROUGE-L / token F1 with Assignment 1 (**ROUGE-L 0.190, token F1 0.280**). Small differences are normal, because QLoRA training is not bit-for-bit deterministic.

In [10]:
def words(t):
    return re.findall(r"[a-z0-9]+", t.lower())

def lcs_len(a, b):
    prev = [0] * (len(b) + 1)
    for x in a:
        cur = [0]
        for j, y in enumerate(b):
            cur.append(prev[j] + 1 if x == y else max(prev[j + 1], cur[-1]))
        prev = cur
    return prev[-1]

def rouge_l(pred, ref):
    p, r = words(pred), words(ref)
    l = lcs_len(p, r) if p and r else 0
    return 0.0 if l == 0 else 2 * (l / len(p)) * (l / len(r)) / (l / len(p) + l / len(r))

def token_f1(pred, ref):
    p, r = collections.Counter(words(pred)), collections.Counter(words(ref))
    common = sum((p & r).values())
    return 0.0 if common == 0 else 2 * (common / sum(p.values())) * (common / sum(r.values())) / (
        common / sum(p.values()) + common / sum(r.values()))

@torch.no_grad()
def generate_answers(model, prompts, max_new_tokens=160, batch_size=8):
    tokenizer.padding_side = "left"
    outs = []
    for i in range(0, len(prompts), batch_size):
        enc = tokenizer(prompts[i:i + batch_size], return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
        gen = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, repetition_penalty=1.1,
                             eos_token_id=tokenizer.eos_token_id, pad_token_id=tokenizer.pad_token_id)
        for row in gen[:, enc["input_ids"].shape[1]:]:
            outs.append(re.split(r"\n\s*###", tokenizer.decode(row, skip_special_tokens=True))[0].strip())
    tokenizer.padding_side = "right"
    return outs

eval_set = eval_pairs[:4] if SMOKE_TEST else eval_pairs
a1_model = PeftModel.from_pretrained(load_model(CPT_MERGED_DIR), SFT_ADAPTER_DIR)
a1_model.config.use_cache = True
t0 = time.time()
answers = generate_answers(a1_model, [build_prompt(p["instruction"]) for p in eval_set],
                           max_new_tokens=32 if SMOKE_TEST else 160)
metrics["rougeL"] = sum(rouge_l(a, p["response"]) for a, p in zip(answers, eval_set)) / len(eval_set)
metrics["token_f1"] = sum(token_f1(a, p["response"]) for a, p in zip(answers, eval_set)) / len(eval_set)
print(f"{len(eval_set)} answers in {time.time() - t0:.0f}s | ROUGE-L {metrics['rougeL']:.3f} (A1: 0.190) | "
      f"token F1 {metrics['token_f1']:.3f} (A1: 0.280)")
for p, a in list(zip(eval_set, answers))[:3]:
    print(f"\nQ: {p['instruction']}\nREFERENCE: {p['response']}\nMODEL:     {a}")
del a1_model; free_memory()

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

39 answers in 59s | ROUGE-L 0.187 (A1: 0.190) | token F1 0.281 (A1: 0.280)

Q: Summarise the four core ideals that the Preamble promises to secure for citizens.
REFERENCE: The Preamble promises citizens justice in social, economic and political life, freedom to think, speak, believe and worship as they choose, equal standing and equal chances in life, and a spirit of fraternity that protects each person's dignity while keeping the nation united.
MODEL:     The Preamble begins with a declaration of the eternal nature of democracy, freedom and justice, and ends with a pledge to safeguard the fundamental rights of all citizens. The first three express the aspirations of every Indian towards a better life under our Union, while Article 37 guarantees equality before and under the law throughout the country.

Q: Why does a law that creates a new State not need the special amendment procedure of Article 368?
REFERENCE: Article 4 says that laws made under Articles 2 and 3, even though they alt

## 5. Save the model description for the RAG notebook

In [11]:
info = {
    "base_model": MODEL_ID,
    "cpt_merged_dir": str(CPT_MERGED_DIR),
    "sft_adapter_dir": str(SFT_ADAPTER_DIR),
    "load_recipe": "AutoModelForCausalLM(cpt_merged_dir, 4-bit NF4) + PeftModel(sft_adapter_dir)",
    "prompt_format": "### System:\\n{system}\\n\\n### Question:\\n{question}\\n\\n### Answer:\\n",
    "system_prompt": SYSTEM,
    "sft_max_len": MAX_LEN,
    "versions": {"transformers": transformers.__version__, "torch": torch.__version__},
    "metrics": metrics,
    "smoke_test": SMOKE_TEST,
}
(MODELS_DIR / "a1_model_info.json").write_text(json.dumps(info, indent=1))
print(json.dumps(info, indent=1))

for p in sorted(MODELS_DIR.rglob("*")):
    if p.is_file():
        print(f"{p.stat().st_size / 1e6:9.1f} MB  {p.relative_to(MODELS_DIR)}")

{
 "base_model": "Qwen/Qwen3-0.6B-Base",
 "cpt_merged_dir": "/content/drive/MyDrive/llm_assignment_2/models/cpt-merged",
 "sft_adapter_dir": "/content/drive/MyDrive/llm_assignment_2/models/sft-on-cpt-adapter",
 "load_recipe": "AutoModelForCausalLM(cpt_merged_dir, 4-bit NF4) + PeftModel(sft_adapter_dir)",
 "prompt_format": "### System:\\n{system}\\n\\n### Question:\\n{question}\\n\\n### Answer:\\n",
 "system_prompt": "You are an expert on the Constitution of India. Answer the question accurately and concisely in your own words.",
 "sft_max_len": 512,
 "versions": {
  "transformers": "5.22.1",
  "torch": "2.14.1+cu130"
 },
 "metrics": {
  "cpt_train_minutes": 5.504069181283315,
  "ppl_heldout_base": 5.457540616337804,
  "ppl_heldout_cpt": 3.4357649358879363,
  "sft_train_minutes": 1.113611110051473,
  "rougeL": 0.18732014167759525,
  "token_f1": 0.28054979015084025
 },
 "smoke_test": false
}
      0.0 MB  a1_model_info.json
      0.0 MB  cpt-adapter/README.md
      0.0 MB  cpt-adapter/ad